# HomeWork #4 - PySpark

Implementazione con PySpark

In [6]:
import csv
import random
from datetime import datetime, timedelta
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, year, split, when, round as spark_round
import time
from pathlib import Path


In [7]:
NUM_EXAMPLES = 200_000

FOLDER_DATASET = "dataset"
FILENAME_DATASET = "pedaggio.csv"

HEADER = [
    "IDVeicolo",
    "TipoVeicolo",
    "Tratta",
    "Pedaggio",
    "DataTransito",
    "FasciaOraria",
    "Provincia",
]

TIPO_VEICOLO = ["AutoTermiche", "AutoElettriche", "Moto", "Camion", "Bus", "Furgone"]

# Mappatura tratta -> province attraversate.
TRATTA_PROVINCE = {
    "A1": ["Milano", "Lodi", "Piacenza", "Parma", "Modena", "Bologna", "Firenze", "Arezzo", "Roma", "Napoli"],
    "A4": ["Torino", "Novara", "Milano", "Bergamo", "Brescia", "Verona", "Vicenza", "Padova", "Venezia", "Trieste"],
    "A7": ["Milano", "Pavia", "Genova"],
    "A10": ["Genova", "Savona", "Imperia"],
    "A12": ["Genova", "La Spezia", "Massa-Carrara", "Livorno", "Roma"],
    "A14": ["Bologna", "Forli-Cesena", "Rimini", "Pesaro-Urbino", "Ancona", "Pescara", "Chieti", "Foggia", "Bari", "Taranto"],
    "A21": ["Torino", "Asti", "Alessandria", "Piacenza", "Brescia"],
    "A22": ["Modena", "Mantova", "Verona", "Trento", "Bolzano"],
    "A23": ["Udine", "Gorizia", "Trieste"],
    "A24": ["Roma", "L'Aquila", "Teramo"],
    "A26": ["Genova", "Alessandria", "Vercelli", "Verbano-Cusio-Ossola"],
    "A30": ["Caserta", "Salerno"],
    "A32": ["Torino"],
    "A90": ["Roma"],
    "A91": ["Roma"],
    "A19": ["Palermo", "Caltanissetta", "Enna", "Catania"],
    "A18": ["Messina", "Catania", "Siracusa"],
    "A20": ["Messina", "Palermo"],
}

TRATTE = list(TRATTA_PROVINCE.keys())

BASE_PEDAGGIO = {
    "AutoTermiche": (8, 22),
    "AutoElettriche": (5, 16),
    "Moto": (4, 14),
    "Camion": (18, 55),
    "Bus": (20, 60),
    "Furgone": (12, 35),
}

ANNO_BASE = 2015
ANNO_FINE = 2025
TASSO_INFLAZIONE_ANNUO = 0.10

def get_fattore_inflazione(anno):
    return (1 + TASSO_INFLAZIONE_ANNUO) ** (anno - ANNO_BASE)

def get_pedaggio(tipo_veicolo, tratta, anno):
    minimo, massimo = BASE_PEDAGGIO[tipo_veicolo]
    fattore_tratta = max(1, len(TRATTA_PROVINCE[tratta]) // 3)
    pedaggio_base = random.randint(minimo, massimo) + fattore_tratta
    pedaggio = pedaggio_base * get_fattore_inflazione(anno)
    return round(pedaggio, 2)

def get_data_transito():
    start = datetime(ANNO_BASE, 1, 1)
    end = datetime(ANNO_FINE, 12, 31, 23, 59, 59)
    delta = end - start
    random_seconds = random.randint(0, int(delta.total_seconds()))
    data_transito = start + timedelta(seconds=random_seconds)
    return data_transito, data_transito.strftime("%Y-%m-%d %H:%M:%S")

def get_fascia_oraria(data_transito):
    ora = data_transito.hour
    if ora < 6:
        return "00:00-06:00"
    if ora < 12:
        return "06:00-12:00"
    if ora < 18:
        return "12:00-18:00"
    return "18:00-24:00"

def genera_record(index):
    tipo_veicolo = random.choice(TIPO_VEICOLO)
    tratta = random.choice(TRATTE)
    provincia = random.choice(TRATTA_PROVINCE[tratta])
    data_transito, data_transito_str = get_data_transito()
    return {
        "IDVeicolo": f"V{index:06d}",
        "TipoVeicolo": tipo_veicolo,
        "Tratta": tratta,
        "Pedaggio": get_pedaggio(tipo_veicolo, tratta, data_transito.year),
        "DataTransito": data_transito_str,
        "FasciaOraria": get_fascia_oraria(data_transito),
        "Provincia": provincia,
    }

dataset_path = Path(FOLDER_DATASET) / FILENAME_DATASET
print(f"Dataset path: {dataset_path}")


if dataset_path.exists():
    print(f"Vuoi rigenerare il dataset? (Attualmente: {FILENAME_DATASET}) [y/N]")

print(f"Vuoi generare il dataset? (Attualmente: {FILENAME_DATASET}) [y/N]")
if input().strip().lower() == 'y':
    with dataset_path.open("w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=HEADER, delimiter=";")
        writer.writeheader()
        for index in range(1, NUM_EXAMPLES + 1):
            writer.writerow(genera_record(index))
    print(f"Dataset generated: {NUM_EXAMPLES} righe")

Dataset path: dataset\pedaggio.csv
Vuoi rigenerare il dataset? (Attualmente: pedaggio.csv) [y/N]
Vuoi generare il dataset? (Attualmente: pedaggio.csv) [y/N]


In [8]:
#Configurazione spark
try:
    spark.stop()
except:
    pass

spark = (
    SparkSession.builder
    .appName("PedaggioAnalysis")
    .master("local[*]")
    .getOrCreate()
)

df = (
    spark.read
    .option("header", "true")
    .option("delimiter", ";")
    .option("inferSchema", "true")
    .csv(str(dataset_path))
)

df = df.withColumn("Anno", year(col("DataTransito")))
df.printSchema()
df.show(5, truncate=False)

root
 |-- IDVeicolo: string (nullable = true)
 |-- TipoVeicolo: string (nullable = true)
 |-- Tratta: string (nullable = true)
 |-- Pedaggio: double (nullable = true)
 |-- DataTransito: timestamp (nullable = true)
 |-- FasciaOraria: string (nullable = true)
 |-- Provincia: string (nullable = true)
 |-- Anno: integer (nullable = true)

+---------+------------+------+--------+-------------------+------------+---------+----+
|IDVeicolo|TipoVeicolo |Tratta|Pedaggio|DataTransito       |FasciaOraria|Provincia|Anno|
+---------+------------+------+--------+-------------------+------------+---------+----+
|V000001  |Moto        |A23   |33.01   |2024-04-06 04:27:17|00:00-06:00 |Trieste  |2024|
|V000002  |Moto        |A32   |20.5    |2019-07-26 00:14:32|00:00-06:00 |Torino   |2019|
|V000003  |Camion      |A91   |40.7    |2016-08-29 10:44:16|06:00-12:00 |Roma     |2016|
|V000004  |Camion      |A12   |59.66   |2025-04-15 07:27:21|06:00-12:00 |Roma     |2025|
|V000005  |AutoTermiche|A20   |29.23   |

In [9]:
print("--- AVVIO ANALISI PYSPARK ---\n")
print("Sessione Spark avviata correttamente.")
print("Esecuzione in corso...")

start_time = time.time()

df = (
    spark.read
    .option("delimiter", ";")
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(dataset_path))
)

df = df.withColumn("Anno", split(col("DataTransito"), "-").getItem(0).cast("int"))

res = (
    df.filter(col("Anno").isin(2015, 2025))
    .groupBy("TipoVeicolo")
    .agg(
        spark_round(avg(when(col("Anno") == 2015, col("Pedaggio"))), 2).alias("Media_2015"),
        spark_round(avg(when(col("Anno") == 2025, col("Pedaggio"))), 2).alias("Media_2025")
    )
    .withColumn("Variazione", spark_round(col("Media_2025") - col("Media_2015"), 2))
    .withColumn(
        "Variazione_percentuale",
        spark_round(((col("Media_2025") - col("Media_2015")) / col("Media_2015")) * 100, 2)
    )
)

righe_calcolate = res.count()

elapsed_time = time.time() - start_time

print("\nJob completato")
print(f"Tempo effettivo di calcolo PySpark: {elapsed_time:.4f} secondi")
print("=" * 70)

res.orderBy("TipoVeicolo").show(truncate=False)

--- AVVIO ANALISI PYSPARK ---

Sessione Spark avviata correttamente.
Esecuzione in corso...

Job completato
Tempo effettivo di calcolo PySpark: 1.1845 secondi
+--------------+----------+----------+----------+----------------------+
|TipoVeicolo   |Media_2015|Media_2025|Variazione|Variazione_percentuale|
+--------------+----------+----------+----------+----------------------+
|AutoElettriche|11.85     |30.67     |18.82     |158.82                |
|AutoTermiche  |16.17     |42.25     |26.08     |161.29                |
|Bus           |41.38     |107.91    |66.53     |160.78                |
|Camion        |37.75     |98.58     |60.83     |161.14                |
|Furgone       |24.79     |64.08     |39.29     |158.49                |
|Moto          |10.3      |26.9      |16.6      |161.17                |
+--------------+----------+----------+----------+----------------------+

